# Random Forests — Bagging, OOB Intuition, and Feature Importance

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb12_random_forests_importance.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how **bootstrap aggregation (bagging)** plus **random feature subsets** turn high-variance single trees into stable ensembles, on both classification and regression spines.
2. Fit `RandomForestClassifier` and `RandomForestRegressor` and demonstrate their CV-score lift over the single tree from nb11 and the **Week-2 reference** baselines.
3. Tune `n_estimators` and `max_features` under the same one-standard-error rule used in nb11.
4. Read the **Out-of-Bag (OOB) score** as a free, non-redundant validation signal — and know when to trust it vs cross-validation.
5. Build the **four-method feature-importance reconciliation table** (linear-coefficient / impurity / permutation / drop-column) — the course-wide reference table that nb15 relies on for interpretation.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — Exercise 1 on the classification track (RF tuning) and Exercise 2 on the regression track (RF tuning). Complete both before submitting your notebook.

---

## 💼 Why This Matters

The single decision tree from nb11 is high-variance: a small change in the training data flips the root split, and from there the entire tree restructures. Both the State Health Department's review board and HomeValue Analytics' deployment council asked the same follow-up question after seeing the tree: *"What happens to this model if you retrain it on next month's data?"*

The honest answer is *"a different tree"*, which is not the answer either stakeholder wanted. **Random forests** fix this by training many trees on bootstrap samples with random feature subsets, then averaging their predictions. No single tree dominates; the forest votes. Variance drops sharply, predictions stabilize, and you keep the ability to handle non-linearity that linear baselines cannot.

The cost is a loss of single-path interpretability — you can no longer trace a flowchart from root to leaf because there are 100 of them. In exchange you get:

- **Stable predictions** under retraining (bias unchanged, variance roughly cut by `1/n_estimators`).
- **Built-in OOB validation** — every tree was trained on a bootstrap sample, so the ~37% of training data left out of each tree is a free held-out set.
- **Four different views of feature importance** that, when they disagree, tell you something interesting about the data.

By the end of today, both the State Health Department's screening forest and HomeValue's price-prediction forest should beat their respective **Week-2 references** by a CI-clear margin. If they do not, the random forest has not earned its added complexity, and the linear baseline ships. That CI-clear discipline is the bar nb14's selection ceremony will enforce on a five-candidate field per spine.

> **A question that often comes up here:** *"if a forest is just an average of trees, why do I need the four-method importance table?"* Because the average of trees is opaque. You cannot read 100 flowcharts. The importance table is the diagnostic that lets you answer the **"what is the model paying attention to?"** question without re-deriving it from the trees. Section 6 makes that table; nb15 lifts it into the M3 milestone.

---

## 1. Setup — Imports, References, Helpers

The setup cell does five things at once: imports the random-forest estimators for both spines, locks `RANDOM_SEED = 474`, defines the **Week-2 reference pipelines** (`reference_clf` = LogReg(C=1.0); `reference_reg` = OLS) carried over from nb09, and registers the plot helpers nb11 introduced (`plot_train_val_curve`, `plot_predicted_vs_actual`) plus three new ones for this notebook (`plot_cv_ci`, `plot_importance_bars`, `plot_importance_heatmap`).

> 💡 **Gemini Prompt:** "Set up imports for sklearn RandomForestClassifier, RandomForestRegressor, DecisionTreeClassifier, DecisionTreeRegressor, LogisticRegression, LinearRegression, permutation_importance, cross_val_score, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing, StandardScaler, Pipeline. Set RANDOM_SEED = 474. Define reference_clf and reference_reg as the Week-2 baseline pipelines. Define helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci (dot plot with 95% CI bars across N models), plot_importance_bars (horizontal bar chart with optional error bars), plot_importance_heatmap (heatmap of feature ranks across methods)."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474`, `reference_clf` and `reference_reg` defined
> - [ ] All five plot helpers callable
> - [ ] No import errors


In [ ]:
# Setup — imports, seed, Week-2 references, plot helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

# --- Course color convention ---
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'

# --- Week-2 reference models (from nb09's CI-overlap test) ---
reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

# --- Helper 1 (from nb11): train-vs-CV overfitting curve ---
def plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax,
                         color_train=GREY, color_val=CLF_COLOR):
    xs = list(range(len(x_values)))
    ax.plot(xs, train, marker='o', label='Train', linewidth=2, color=color_train)
    ax.errorbar(xs, val_mean, yerr=val_std, marker='s', label='5-fold CV ± SD',
                linewidth=2, capsize=5, color=color_val)
    ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in x_values])
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

# --- Helper 2 (from nb11): predicted-vs-actual scatter ---
def plot_predicted_vs_actual(y_true, y_pred, ax, title='Predicted vs Actual',
                             color=REG_COLOR):
    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=color)
    lo, hi = float(min(np.min(y_true), np.min(y_pred))), float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

# --- Helper 3 (NEW): CV-CI dot plot (with 95% CI from Student's t) ---
def plot_cv_ci(scores_dict, metric_name, title, ax, color=CLF_COLOR, k=5):
    """scores_dict: {model_name: array of fold scores}; draws 95% CI from t-dist."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    rows = []
    for name, scores in scores_dict.items():
        m = float(np.mean(scores)); sd = float(np.std(scores, ddof=1))
        rows.append({'name': name, 'mean': m, 'half_w': t_crit * sd / np.sqrt(k)})
    df = pd.DataFrame(rows).sort_values('mean')
    ax.errorbar(df['mean'], df['name'], xerr=df['half_w'],
                fmt='o', capsize=6, linewidth=2, color=color, markersize=10)
    for _, r in df.iterrows():
        ax.text(r['mean'] + r['half_w'] + (r['half_w']*0.2 if r['half_w']>0 else 0.001),
                r['name'], f"{r['mean']:.4f}", va='center', fontsize=9)
    ax.set_xlabel(f'5-fold CV {metric_name} (mean ± 95% CI)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Helper 4 (NEW): horizontal importance bar chart with optional error bars ---
def plot_importance_bars(importances, names, ax, errors=None, color=CLF_COLOR,
                         title='Feature importance', top_n=15):
    """Horizontal bar chart, sorted descending. errors optional (e.g., permutation SD)."""
    df = pd.DataFrame({'name': names, 'imp': importances})
    if errors is not None: df['err'] = errors
    df = df.sort_values('imp', ascending=True).tail(top_n)
    if errors is not None:
        ax.barh(df['name'], df['imp'], xerr=df['err'], color=color, edgecolor='black', capsize=4)
    else:
        ax.barh(df['name'], df['imp'], color=color, edgecolor='black')
    ax.set_xlabel('Importance')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Helper 5 (NEW): rank-disagreement heatmap (4-method reconciliation) ---
def plot_importance_heatmap(rank_df, ax, title='Feature rank across methods', top_n=15):
    """rank_df: rows = features, columns = methods; values = rank (1 = most important)."""
    # Show only top_n features by best (smallest) rank across methods
    best_rank = rank_df.min(axis=1)
    keep = best_rank.nsmallest(top_n).index
    sub = rank_df.loc[keep].sort_values(rank_df.columns[0])
    im = ax.imshow(sub.values, cmap='RdYlGn_r', aspect='auto', vmin=1, vmax=rank_df.shape[0])
    ax.set_xticks(range(sub.shape[1])); ax.set_xticklabels(sub.columns, rotation=20, ha='right')
    ax.set_yticks(range(sub.shape[0])); ax.set_yticklabels(sub.index)
    for i in range(sub.shape[0]):
        for j in range(sub.shape[1]):
            ax.text(j, i, int(sub.values[i, j]), ha='center', va='center', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Rank (1 = most important)', shrink=0.7)

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print(f"✓ Week-2 references: reference_clf, reference_reg")
print(f"✓ Helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci, "
      f"plot_importance_bars, plot_importance_heatmap")


**Reading the output:**

Five plot helpers are now in scope. Two carry over from nb11 (`plot_train_val_curve`, `plot_predicted_vs_actual`). Three are new for this notebook: `plot_cv_ci` produces the dot-plot-with-error-bars that you will see in Section 8's comprehensive comparison and again in nb14's selection ceremony; `plot_importance_bars` standardizes the horizontal feature-importance chart used in Sections 6 and 7; `plot_importance_heatmap` renders the four-method reconciliation table — the visualization that turns rank disagreement across methods into the central pedagogical payload of Section 6.

The two reference pipelines `reference_clf` and `reference_reg` are exactly what survived nb09's CI-overlap test on the two datasets. They will appear in Section 8 as the lower bars on the comprehensive-comparison plot — the floor every Random Forest variant has to clear by a CI-clear margin.

**Key takeaway:** All five helpers + both Week-2 references are defined once, reused across every section. Helper definitions never change between notebooks; their signatures stay stable so muscle memory transfers when you reuse them in your own M3 work.

---

## 2. Load Both Datasets — Three Holdout Splits, Two Locked Envelopes

Same 60/20/20 locking discipline you set up in nb11. Both spines do the canonical two-step split with `random_state=RANDOM_SEED` — 60% training, 20% validation, 20% sealed test — and the test envelopes stay sealed until nb14's ceremony. Classification uses stratified CV; regression uses plain `KFold`. The variable suffixes `_clf` / `_reg` keep the two namespaces unconfusable.

In [ ]:
# --- Classification track: Wisconsin breast cancer ---
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target

# 60/20/20: first carve off 20% test, then split the remaining 80% into 75/25 → 60/20.
X_clf_temp, X_test_clf, y_clf_temp, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# --- Regression track: California Housing ---
data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target

X_reg_temp, X_test_reg, y_reg_temp, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("=== CLASSIFICATION (Wisconsin Breast Cancer) ===")
print(f"  Train: {len(X_train_clf):>6} | Val: {len(X_val_clf):>5} | Test: {len(X_test_clf):>5} (LOCKED until nb14)")
print()
print("=== REGRESSION (California Housing) ===")
print(f"  Train: {len(X_train_reg):>6} | Val: {len(X_val_reg):>5} | Test: {len(X_test_reg):>5} (LOCKED until nb14)")


**Reading the output:**

Same 60/20/20 split as nb11 — the same `random_state=RANDOM_SEED` produces the same partition, so the CV scores you compute here are directly comparable to nb11's single-tree numbers. That is the whole point of locking the seed: when nb14's ceremony declares a champion, the chain of CV evidence from nb11 → nb12 → nb13 → nb14 sits on identical splits.

**Key takeaway:** Same splits as nb11 → results are directly comparable. From here, every model's CV mean can be quoted alongside nb11's tree numbers without an asterisk.

---

## 3. From Single Tree to Forest — The Bagging Idea

nb11 ended with an honest finding: a single decision tree is interpretable but **high-variance**. Train it on a slightly different set of patients or census tracts and the root split can flip to a different feature, the leaves rearrange, and the predictions move. That fragility is the problem random forests solve, and the fix has a beautifully simple shape borrowed from a technique you may already know from finance: **average many noisy estimates, and the noise cancels out**.

Imagine a sell-side analyst asking ten of her colleagues for their 12-month price target on a stock. Each colleague has their own model, their own data slice, their own biases. Any one estimate is noisy. But if she takes the **average** across all ten, the analyst-specific noise tends to cancel, and the average sits closer to the true value than almost any individual estimate. That is the entire idea behind **bagging** — short for *bootstrap aggregating* — applied to decision trees. Each "analyst" in the forest is one tree, trained on a slightly different sample of the data, and the forest's prediction is the average (for regression) or majority vote (for classification) across all the trees.

The "slightly different sample" part is what bootstrap does. The figure below is the canonical bootstrap-resampling diagram from the ISLR textbook:

<center>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/5_11-1.png" width="640" alt="Bootstrap resampling — draw B samples with replacement from the original training data, fit one model per sample, aggregate the B estimates.">
</center>

The original training data on the left has three rows. To create one **bootstrap sample**, draw three rows *with replacement* — each draw is independent, and the same row can be picked more than once. In the top sample (Z*¹) row 3 was drawn twice and row 2 was never drawn. The middle sample (Z*²) drew rows 2, 3, and 1 — every original row appears exactly once. The bottom sample (Z*ᴮ) drew row 2 twice and row 1 once. Each sample has the same size as the original training set, but the *composition* is different. Each sample produces its own estimate `α̂*ᵇ`; the final ensemble combines all `B` of them.

Two numbers fall out of this procedure that are worth memorizing:

- **About 63% of the original rows appear** in any one bootstrap sample (some appear multiple times, others not at all).
- **The remaining ~37% are out-of-bag** for that sample — they form a free held-out set you can use to validate the tree without spending any of the locked test data. Section 6 will turn this into the OOB diagnostic.

In bagging we draw `B` bootstrap samples — typically 100 to 500 — fit one decision tree on each, and combine their predictions at the end. The intuition the financial analyst used scales directly: averaging `B` noisy estimators reduces the variance of the ensemble's prediction far below the variance of any individual tree. Trees fit on different bootstraps are not perfectly independent (they share most of the original rows), so the variance does not fall as fast as the textbook `1/B` formula would suggest — but the direction is always the same. More bootstraps means less wobble.

The plot below makes the bootstrap part visible on real data. Five bootstrap samples are drawn from the breast-cancer **training set you just loaded in Section 2**, and the distribution of one feature (`mean radius`) is plotted under each. The five curves cluster around the original distribution but diverge in details — that variability across bootstraps is exactly what averaging a forest cancels out.

In [ ]:
# Bootstrap KDEs — visualize what each base learner actually sees on real data.
# Uses X_train_clf['mean radius'] from the 341-patient training pool loaded in Section 2.
X_show = X_train_clf['mean radius'].values
rng = np.random.default_rng(RANDOM_SEED)

fig, ax = plt.subplots(figsize=(11, 5))
xs = np.linspace(X_show.min(), X_show.max(), 400)
for b in range(5):
    boot = rng.choice(X_show, size=len(X_show), replace=True)
    bw = 1.06 * boot.std() * len(boot) ** (-1/5)
    kde = np.exp(-0.5 * ((xs[:, None] - boot[None, :]) / bw) ** 2).sum(axis=1) / (len(boot) * bw * np.sqrt(2*np.pi))
    ax.plot(xs, kde, alpha=0.7, linewidth=2, label=f'Bootstrap {b+1}')

bw0 = 1.06 * X_show.std() * len(X_show) ** (-1/5)
kde0 = np.exp(-0.5 * ((xs[:, None] - X_show[None, :]) / bw0) ** 2).sum(axis=1) / (len(X_show) * bw0 * np.sqrt(2*np.pi))
ax.plot(xs, kde0, color='black', linewidth=3, linestyle='--', label='Original training data', alpha=0.8)

ax.set_xlabel('mean radius')
ax.set_ylabel('density')
ax.set_title('Five bootstrap samples of the breast-cancer training set — every base learner sees a slightly different slice',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Each bootstrap is similar but not identical — a different tree fits a different shape.")
print("✓ Random forests average across dozens to hundreds of these slices; the forest's variance is far below any single tree's.")


**Reading the output:**

All five bootstrap curves cluster around the original training distribution (the dashed black line) but diverge in details — different peaks, different tails, different local densities. A decision tree fit on each bootstrap will pick slightly different split thresholds in `mean radius` and slightly different child structures, even though the underlying training set is the same 341-patient pool you loaded in Section 2.

The variance you see across these five curves is the variance you saw in nb11's `max_depth=20` overfit row — large, fold-dependent, fragile under retraining. The random forest's prediction is the average (for regression) or majority vote (for classification) over **dozens or hundreds** of trees like these. Two trees that disagree on a borderline patient cancel each other out at the vote; two trees that both vote malignant on an obvious case reinforce each other. The averaging mechanism turns the ensemble into a much smoother, much more stable predictor than any single tree.

> **A question that often comes up here:** *"if bootstrap is just sampling with replacement, why does that produce different trees?"* Two reasons. First, each tree's training set is genuinely different — fit one tree on the same data twice and you get the same tree, but fit it on two different bootstraps and you get two different trees, with different root splits, different thresholds, and different leaves. Second, the trees' errors are partially **uncorrelated** — when tree A is wrong on a particular patient, tree B is often right, and the average of their predictions is closer to truth than either individual prediction. That is the same variance-reduction-by-averaging logic the financial analyst used at the top of this section, applied to a model ensemble instead of a panel of forecasters.

**Key takeaway:** The bootstrap creates the diversity; the averaging cashes the diversity in as variance reduction. Section 4 measures the cash-in directly — single tree's CV variability versus forest's CV variability under identical conditions, with the error bar shrinking visibly when you switch from one tree to one hundred.

---

## 4. Single Tree vs Random Forest — Paired

The headline comparison: each spine's **best single tree from nb11** versus a 100-tree random forest fit at the *same depth*. Classification uses `max_depth=3` (nb11's §7 winner — the depth the State Health Department's review board can read on a single page); regression uses `max_depth=5` (nb11's §7 winner — the depth that balanced bias and variance on California Housing without crushing CV runtime). Same depth, same data, same CV folds — the only thing that changes is *one tree → one hundred trees averaged*. The forest should beat the tree on **both** mean score (modestly) and CV standard deviation (substantially) — the latter is the variance-reduction payoff from Section 3 finally made visible on real CV folds.

> 💡 **Gemini Prompt:** "Fit DecisionTreeClassifier(max_depth=3, random_state=474) and RandomForestClassifier(n_estimators=100, max_depth=3, random_state=474, n_jobs=-1) on X_train_clf, evaluate via 5-fold CV ROC-AUC. Same comparison on the regression spine: DecisionTreeRegressor(max_depth=5, random_state=474) vs RandomForestRegressor(n_estimators=100, max_depth=5, random_state=474, n_jobs=-1) with R². Report mean and SD for both spines; build a 1×2 CV-CI dot plot using the plot_cv_ci helper."
>
> **After running, verify:**
> - [ ] Forest CV mean > tree CV mean on both spines
> - [ ] Forest CV SD < tree CV SD on both spines (the variance-reduction signal)
> - [ ] Both panels use the plot_cv_ci helper (95% CI from Student's t)


In [ ]:
# Single tree vs forest — paired comparison.
# Depths match nb11's §7 head-to-head: depth=3 for classification, depth=5 for regression.
tree_clf   = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
forest_clf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_SEED, n_jobs=-1)
tree_reg   = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED)
forest_reg = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1)

clf_scores = {
    'Single Tree (depth=3)':  cross_val_score(tree_clf,   X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (100×3)':  cross_val_score(forest_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_scores = {
    'Single Tree (depth=5)':  cross_val_score(tree_reg,   X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
    'Random Forest (100×5)':  cross_val_score(forest_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
}

k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)
def _print_ci_summary(scores_dict, title):
    rows = []
    for name, s in scores_dict.items():
        mean = float(s.mean()); sd = float(s.std(ddof=1))
        half_w = t_crit * sd / np.sqrt(k)
        rows.append({'model': name, 'mean': mean, 'sd': sd, 'half_w': half_w,
                     'ci_low': mean - half_w, 'ci_high': mean + half_w})
    print(title)
    print(pd.DataFrame(rows).to_string(index=False))

_print_ci_summary(clf_scores, "=== CLASSIFICATION (5-fold CV ROC-AUC, mean ± 95% CI) — depth=3 on both ===")
print()
_print_ci_summary(reg_scores, "=== REGRESSION (5-fold CV R², mean ± 95% CI) — depth=5 on both ===")

# Side-by-side CV-CI dot plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_cv_ci(clf_scores, 'ROC-AUC', 'Classification — single tree vs forest (depth=3)', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_scores, 'R²',      'Regression — single tree vs forest (depth=5)',     axes[1], color=REG_COLOR)
fig.suptitle('Variance reduction in action — forest CIs are tighter than the single tree on both spines',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On both spines the random forest beats the single tree on **both** mean score and SD, holding nb11's chosen depth fixed (3 for classification, 5 for regression). The mean lift is meaningful — typically a handful of score points on these datasets (around 5–6 points on both spines for this seed) — single trees at their well-chosen depth are competitive, but the forest still picks up a clear improvement. The SD reduction, on the other hand, is dramatic: the forest's 95% CI is typically about half the width of the single tree's. That tightening is exactly the variance-reduction-by-averaging mechanic from Section 3 made visible on real CV folds.

> **A question that often comes up here:** *"if 100 trees average to a slightly better mean than one tree, why not 1000 trees?"* Two reasons. First, the marginal CV gain from each additional tree shrinks fast — Section 5 will show the curve plateauing around 100 to 200 trees on both spines, with essentially no incremental score after that point. Second, the variance-reduction math has a hard floor: when bootstrap samples are highly correlated (which they are, by construction — they share most of the original rows), the realized reduction is less than `1/B` and asymptotes to a non-zero residual. Past that asymptote, more trees just buy you fit time, not predictive lift. Section 5 also covers `max_features`, which is the lever that controls how correlated the trees are — and therefore how low the asymptote sits.

**Key takeaway:** Forests beat trees on both mean and SD, but the bigger win is the variance reduction (tighter CI) rather than the mean lift. That stability is what makes a forest defensible to stakeholders who will retrain monthly — the worst-case fold-to-fold swing is much smaller.

---

## 5. Tuning Random Forests — `n_estimators` and `max_features`

Random forests have two key hyperparameters beyond what trees have: **`n_estimators`** (how many trees in the ensemble) and **`max_features`** (how many features each split considers). Both shape the bias/variance trade-off.

- More `n_estimators` → lower variance, eventual plateau, no risk of overfitting per se (more trees just smooth the average).
- Smaller `max_features` → more diverse trees → lower correlation → larger variance reduction at the cost of some single-tree quality. The sklearn defaults are `sqrt(n_features)` for classification and `1.0` (all features) for regression — the latter is often suboptimal and worth tuning.

Two paired sweeps: one for `n_estimators`, one for `max_features`. Both use 3-fold CV here (instead of 5) to keep the regression-grid runtime manageable on Colab; the headline plots in later sections will use full 5-fold CV.

> 💡 **Gemini Prompt:** "Sweep n_estimators in [10, 25, 50, 100, 200] for both spines using 3-fold CV (scoring='roc_auc' for clf, 'r2' for reg). Then sweep max_features in ['sqrt', 'log2', 0.3, 0.5, None] for both spines at n_estimators=100. Two paired figures using plot_train_val_curve for the n_estimators sweep, plus a side-by-side bar plot for max_features."
>
> **After running, verify:**
> - [ ] n_estimators curves both plateau (clf around 50–100, reg around 100)
> - [ ] max_features sweep shows that 'sqrt'/0.3 typically beat 'None' on regression
> - [ ] All cells use n_jobs=-1


In [ ]:
# Use 3-fold CV for these inline sweeps to keep regression runtime modest on Colab
cv_clf_3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
cv_reg_3 = KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

# --- Sweep 1: n_estimators ---
n_est_grid = [10, 25, 50, 100, 200]

clf_train, clf_val_mean, clf_val_std = [], [], []
for n in n_est_grid:
    m = RandomForestClassifier(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train_clf, y_train_clf)
    clf_train.append(m.score(X_train_clf, y_train_clf))
    s = cross_val_score(m, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='roc_auc', n_jobs=-1)
    clf_val_mean.append(s.mean()); clf_val_std.append(s.std(ddof=1))

reg_train, reg_val_mean, reg_val_std = [], [], []
for n in n_est_grid:
    m = RandomForestRegressor(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train_reg, y_train_reg)
    reg_train.append(m.score(X_train_reg, y_train_reg))
    s = cross_val_score(m, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
    reg_val_mean.append(s.mean()); reg_val_std.append(s.std(ddof=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_train_val_curve(n_est_grid, clf_train, clf_val_mean, clf_val_std,
                     'n_estimators', 'Accuracy (train) / ROC-AUC (CV)',
                     'Classification', axes[0], color_val=CLF_COLOR)
plot_train_val_curve(n_est_grid, reg_train, reg_val_mean, reg_val_std,
                     'n_estimators', 'R² (train) / R² (CV)',
                     'Regression', axes[1], color_val=REG_COLOR)
fig.suptitle('CV score plateaus as n_estimators grows — diminishing returns past ~100 trees',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# --- Sweep 2: max_features at n_estimators=100 ---
max_feat_grid = ['sqrt', 'log2', 0.3, 0.5, None]

k_mf = 3  # 3-fold CV for this inline sweep
t_crit_mf = stats.t.ppf(0.975, df=k_mf - 1)

clf_mf_means, clf_mf_sds, clf_mf_half = [], [], []
reg_mf_means, reg_mf_sds, reg_mf_half = [], [], []
for mf in max_feat_grid:
    m_c = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=RANDOM_SEED, n_jobs=-1)
    s_c = cross_val_score(m_c, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='roc_auc', n_jobs=-1)
    sd_c = s_c.std(ddof=1)
    clf_mf_means.append(s_c.mean()); clf_mf_sds.append(sd_c)
    clf_mf_half.append(t_crit_mf * sd_c / np.sqrt(k_mf))

    m_r = RandomForestRegressor(n_estimators=100, max_features=mf, random_state=RANDOM_SEED, n_jobs=-1)
    s_r = cross_val_score(m_r, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
    sd_r = s_r.std(ddof=1)
    reg_mf_means.append(s_r.mean()); reg_mf_sds.append(sd_r)
    reg_mf_half.append(t_crit_mf * sd_r / np.sqrt(k_mf))

mf_df = pd.DataFrame({
    'max_features': [str(v) for v in max_feat_grid],
    'clf_mean':     clf_mf_means,  'clf_ci_low':  [m - h for m, h in zip(clf_mf_means, clf_mf_half)],
    'clf_ci_high':  [m + h for m, h in zip(clf_mf_means, clf_mf_half)],
    'reg_mean':     reg_mf_means,  'reg_ci_low':  [m - h for m, h in zip(reg_mf_means, reg_mf_half)],
    'reg_ci_high':  [m + h for m, h in zip(reg_mf_means, reg_mf_half)],
})
print('=== MAX_FEATURES SWEEP — 3-fold CV mean and 95% CI (Student\'s t, df=2) ===')
print(mf_df.to_string(index=False))

labels = [str(v) for v in max_feat_grid]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.bar(labels, clf_mf_means, yerr=clf_mf_half, capsize=8, color=CLF_COLOR, edgecolor='black')
for i, m in enumerate(clf_mf_means):
    ax.text(i, m + clf_mf_half[i] + 0.003, f'{m:.4f}', ha='center', fontsize=10)
ax.set_xlabel('max_features'); ax.set_ylabel('3-fold CV ROC-AUC (mean ± 95% CI)')
ax.set_title('Classification — max_features sweep', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(labels, reg_mf_means, yerr=reg_mf_half, capsize=8, color=REG_COLOR, edgecolor='black')
for i, m in enumerate(reg_mf_means):
    ax.text(i, m + reg_mf_half[i] + 0.003, f'{m:.4f}', ha='center', fontsize=10)
ax.set_xlabel('max_features'); ax.set_ylabel('3-fold CV R² (mean ± 95% CI)')
ax.set_title('Regression — max_features sweep', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('max_features controls tree diversity — smaller subsets often beat using all features',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The `n_estimators` curves on both spines tell the same story: a sharp lift from 10 → 50 trees, then a slow plateau past 100. By 200 trees the marginal improvement is in the third decimal place. For the headline plots and the M3 milestone, **`n_estimators=100` is the right working default** — beyond that you spend fit time without buying CV lift.

The `max_features` sweep is more interesting because the right answer is dataset-dependent. On classification, `sqrt(n_features)` (the sklearn default) usually wins or ties. On regression, the sklearn default of `None` (all features) often loses to `sqrt` or `0.3` — using all features at every split makes the trees too similar, which kills the diversity that makes bagging work. Tuning `max_features` to a fraction smaller than 1.0 typically buys 1–2 R² points on California housing.

> **A question that often comes up here:** *"why is the regression default `None` if it's usually wrong?"* Historical reasons. The original Breiman paper proposed `sqrt(p)` for classification and `p/3` for regression; the sklearn `1.0` default is more permissive and matches a different tradition. The point is not to memorize the default but to **always tune `max_features` for regression projects** — a 30-second hyperparameter sweep typically produces a meaningful lift.

**Key takeaway:** `n_estimators=100` is a defensible default; `max_features` always deserves a sweep, especially on the regression spine.

---

## 6. Out-of-Bag (OOB) Score — Free Validation

Section 3 noted that every bootstrap sample leaves about 37% of the training rows un-sampled. Those un-sampled rows are **out-of-bag** for the tree that was fit on that bootstrap — that specific tree has never seen them. That gives the random forest a useful party trick: for any training row, predict it using **only** the trees that did not see it in their bootstrap. The result is a held-out score on the entire training set, computed during fitting, at zero extra cost. That is the **OOB score**.

The OOB number usually agrees with the 5-fold CV mean to within a small fraction of a point. When the two diverge meaningfully — say, more than one CV standard deviation apart — something is wrong with the i.i.d. assumption: typically the data has hidden structure (groups, time, panel effects) that bootstrap sampling does not respect. Neither of today's datasets has that structure, so OOB and CV should track each other closely.

The plot below traces both signals as `n_estimators` grows. The OOB curve is a single value per setting (no folds to average); the CV curve is the 3-fold mean with error bars. They should converge as the forest matures.

In [ ]:
# OOB vs CV — paired
n_est_grid_oob = [25, 50, 100, 200, 300]

k_oob = 3
t_crit_oob = stats.t.ppf(0.975, df=k_oob - 1)

clf_oob, clf_cv_mean, clf_cv_half = [], [], []
for n in n_est_grid_oob:
    m = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=RANDOM_SEED, n_jobs=-1)
    m.fit(X_train_clf, y_train_clf)
    clf_oob.append(m.oob_score_)
    s = cross_val_score(m, X_train_clf, y_train_clf, cv=cv_clf_3, scoring='accuracy', n_jobs=-1)
    clf_cv_mean.append(s.mean())
    clf_cv_half.append(t_crit_oob * s.std(ddof=1) / np.sqrt(k_oob))

reg_oob, reg_cv_mean, reg_cv_half = [], [], []
for n in n_est_grid_oob:
    m = RandomForestRegressor(n_estimators=n, oob_score=True, random_state=RANDOM_SEED, n_jobs=-1)
    m.fit(X_train_reg, y_train_reg)
    reg_oob.append(m.oob_score_)  # OOB R²
    s = cross_val_score(m, X_train_reg, y_train_reg, cv=cv_reg_3, scoring='r2', n_jobs=-1)
    reg_cv_mean.append(s.mean())
    reg_cv_half.append(t_crit_oob * s.std(ddof=1) / np.sqrt(k_oob))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
ax.plot(n_est_grid_oob, clf_oob, marker='o', linewidth=2, color=CLF_COLOR, label='OOB accuracy')
ax.errorbar(n_est_grid_oob, clf_cv_mean, yerr=clf_cv_half, marker='s', linewidth=2,
            capsize=5, color=GREY, label='3-fold CV accuracy (mean ± 95% CI)')
ax.set_xlabel('n_estimators'); ax.set_ylabel('Accuracy')
ax.set_title('Classification — OOB vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(n_est_grid_oob, reg_oob, marker='o', linewidth=2, color=REG_COLOR, label='OOB R²')
ax.errorbar(n_est_grid_oob, reg_cv_mean, yerr=reg_cv_half, marker='s', linewidth=2,
            capsize=5, color=GREY, label='3-fold CV R² (mean ± 95% CI)')
ax.set_xlabel('n_estimators'); ax.set_ylabel('R²')
ax.set_title('Regression — OOB vs CV', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('OOB tracks CV closely — a free validation signal at zero compute cost',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 OOB and CV curves should overlap. If they diverge, suspect data dependence (groups, time).")


**Reading the output:**

On both spines OOB and CV agree closely — they often differ by less than one SD. That agreement is the hallmark of i.i.d. data: when bootstrap sampling is appropriate, OOB and CV are estimating the same quantity. You get the OOB number essentially for free during the forest fit; CV requires `k` separate fits at `k×` the cost.

When does OOB lie? When samples are not independent. The three most common cases in business data are time-series problems (where future samples share a generative process with past ones, so bootstrap mixes time periods and OOB comes back optimistically high), hierarchical or grouped data such as multiple visits per patient or multiple orders per customer (where leaving out one row but keeping other rows from the same group inflates OOB), and panel data with strong fixed effects (the same group-correlation issue under a different name). The unifying pattern is that bootstrap treats rows as exchangeable; the moment your rows are not exchangeable, the 37% you call "held out" is actually contaminated.

For the M3 milestone, OOB is a useful sanity check on top of CV — when both report the same number, you have two estimates with two different sets of assumptions agreeing, which is stronger evidence than either alone.

> **A question that often comes up here:** *"if OOB is free and CV costs 5× more, why ever run CV?"* Three reasons. First, the **statistical machinery** (Student's *t* CIs, the CI-overlap test) is built around CV folds — OOB gives you one number, not a distribution, so you cannot put a CI on it the same way. Second, CV with explicit folds lets you compare different model classes on identical splits, which OOB cannot do — only forests have OOB, so a LogReg or GBM cannot enter the same comparison. Third, CV catches data-dependence problems by failing visibly, while OOB silently agrees with itself even when both estimates are wrong.

**Key takeaway:** OOB is the right second-opinion on CV when both apply; CV is the foundation for cross-model comparison. nb14 will use CV exclusively because the comparison is across model families, not within forests.

---

## 7. Feature Importance — The Four-Method Reconciliation Table

This is the section nb15's interpretation work leans on. The stakeholder question is always the same — *"which features matter most?"* — and the honest answer is that there are at least four reasonable ways to measure that, and they often disagree. The disagreement is not a flaw; it is the pedagogical payload. When all four methods agree on a feature, the answer is unambiguous and you can quote any method to the stakeholder. When they disagree, you have to think about **what each method is actually measuring** before you decide what to report.

The four methods we will reconcile:

| # | Method | What it measures | Cost |
|---|---|---|---|
| 1 | **Linear coefficient magnitude** (Week-2 reference) | Standardized β — change in target per SD change in feature, *holding other features constant* | Free (one model fit) |
| 2 | **Impurity-based MDI** (`feature_importances_`) | Average decrease in node impurity weighted by samples reaching that node, summed across all trees | Free (one forest fit) |
| 3 | **Permutation importance** | Score drop when the feature's values are randomly shuffled in evaluation | Moderate (`n_repeats × n_features` re-evaluations) |
| 4 | **Drop-column importance** | Score drop when the feature is removed and the forest is **refit** without it | Expensive (`n_features` refits) |

The output is a **rank heatmap**: rows are features, columns are methods, cell colour encodes the rank a method assigned to a feature. Rows that are uniformly dark are robustly important across all four methods; rows that are mottled tell you which method's framing dominates the conclusion.

> 💡 **Gemini Prompt:** "Compute four importance methods on the breast cancer training set: standardized linear coefficient magnitudes from reference_clf; MDI from a fitted RandomForestClassifier(200 trees); permutation importance with 10 repeats; drop-column importance via 5-fold CV ROC-AUC for each feature dropped. Build a rank DataFrame (rows = features, columns = methods, values = rank). Render as a heatmap with the plot_importance_heatmap helper. Same four methods on California housing using R² and OLS."
>
> **After running, verify:**
> - [ ] Both spines produce a rank DataFrame with 4 columns (one per method)
> - [ ] Heatmap shows top-15 features per spine sorted by linear-coef rank
> - [ ] At least one feature has wildly different ranks across methods (the disagreement signal)


In [ ]:
# Helper: compute four-method importance ranks for a single spine
def four_method_ranks(reference_pipeline, ref_step_name,
                      forest_estimator,
                      X_train, y_train, scoring, cv, label,
                      drop_col_subset=None):
    """Returns a DataFrame with rows=features, columns=4 methods, values=rank."""
    feat_names = list(X_train.columns)

    # Method 1: linear coefficient magnitude (standardized)
    ref = reference_pipeline.fit(X_train, y_train)
    coef = ref.named_steps[ref_step_name].coef_
    coef = coef.ravel() if coef.ndim > 1 else coef
    coef_mag = np.abs(coef)

    # Method 2: MDI (impurity-based)
    forest = forest_estimator.fit(X_train, y_train)
    mdi = forest.feature_importances_

    # Method 3: permutation importance
    perm = permutation_importance(forest, X_train, y_train,
                                  scoring=scoring, n_repeats=10,
                                  random_state=RANDOM_SEED, n_jobs=-1)
    perm_mean = perm.importances_mean

    # Method 4: drop-column importance — refit without each feature, measure CV score drop
    base_score = cross_val_score(forest_estimator, X_train, y_train,
                                 cv=cv, scoring=scoring, n_jobs=-1).mean()
    drop_imp = []
    cols_to_drop = drop_col_subset if drop_col_subset is not None else feat_names
    drop_set = set(cols_to_drop)
    for col in feat_names:
        if col not in drop_set:
            drop_imp.append(np.nan)
            continue
        X_drop = X_train.drop(columns=[col])
        s = cross_val_score(forest_estimator, X_drop, y_train,
                            cv=cv, scoring=scoring, n_jobs=-1).mean()
        drop_imp.append(base_score - s)
    drop_imp = np.array(drop_imp, dtype=float)

    # Convert each method's importance to RANK (1 = most important).
    # NaN drop-importances (skipped features) get the worst rank.
    def to_rank(imp):
        s = pd.Series(imp, index=feat_names)
        return s.rank(ascending=False, method='min', na_option='bottom').astype(int)

    rank_df = pd.DataFrame({
        'Linear coef': to_rank(coef_mag),
        'MDI':         to_rank(mdi),
        'Permutation': to_rank(perm_mean),
        'Drop-column': to_rank(drop_imp),
    }, index=feat_names)
    print(f"=== {label} — top-10 features by linear coef rank ===")
    print(rank_df.sort_values('Linear coef').head(10).to_string())
    return rank_df, perm

# --- Classification: subset drop-column to the 10 features with highest MDI to keep runtime modest ---
quick_forest_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
mdi_quick = quick_forest_clf.fit(X_train_clf, y_train_clf).feature_importances_
top_clf_features = list(X_train_clf.columns[np.argsort(mdi_quick)[::-1][:10]])

ranks_clf, perm_clf = four_method_ranks(
    reference_pipeline=Pipeline([('scaler', StandardScaler()),
                                 ('clf', LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))]),
    ref_step_name='clf',
    forest_estimator=RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    X_train=X_train_clf, y_train=y_train_clf,
    scoring='roc_auc', cv=cv_clf_3, label='CLASSIFICATION',
    drop_col_subset=top_clf_features
)

# --- Regression: subset drop-column to top 5 by MDI to keep runtime under control ---
quick_forest_reg = RandomForestRegressor(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1)
mdi_quick_r = quick_forest_reg.fit(X_train_reg, y_train_reg).feature_importances_
top_reg_features = list(X_train_reg.columns[np.argsort(mdi_quick_r)[::-1][:5]])

ranks_reg, perm_reg = four_method_ranks(
    reference_pipeline=Pipeline([('scaler', StandardScaler()),
                                 ('reg', LinearRegression())]),
    ref_step_name='reg',
    forest_estimator=RandomForestRegressor(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1),
    X_train=X_train_reg, y_train=y_train_reg,
    scoring='r2', cv=cv_reg_3, label='REGRESSION',
    drop_col_subset=top_reg_features
)


In [ ]:
# Render the four-method rank heatmaps
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
plot_importance_heatmap(ranks_clf, axes[0],
                        title='Classification (Wisconsin Breast Cancer) — feature rank across 4 methods',
                        top_n=15)
plot_importance_heatmap(ranks_reg, axes[1],
                        title='Regression (California Housing) — feature rank across 4 methods',
                        top_n=8)
plt.tight_layout()
plt.show()

print("\n💡 Rows that are uniformly green (low ranks across all 4 methods) are unambiguous winners.")
print("💡 Rows that are mottled (varying ranks) are where methods disagree — investigate why.")


**Reading the output:**

On classification, the top features in MDI, permutation, and drop-column usually agree closely — `worst perimeter`, `worst concave points`, `worst radius`, `mean concave points`, and a few related cell-shape measurements dominate every method. The linear-coefficient rank often differs because logistic regression spreads predictive power across **correlated features** (the "worst" and "mean" versions of the same measurement steal predictive credit from each other), while the forest concentrates importance on whichever variant happened to be picked first in the trees. **That kind of disagreement is correlation talking** — not contradiction.

On regression, the top features (`MedInc`, `AveOccup`, `Latitude`, `Longitude`) usually agree across methods. `MedInc` (median income) is the single overwhelming predictor; the geography features matter but trade ranks across methods because of the same correlation effect. The disagreement worth watching for here is when MDI and permutation **disagree** on a feature with many distinct values — MDI tends to inflate the importance of high-cardinality features (more split candidates means more chances to reduce impurity), while permutation does not. If you ever see a near-categorical feature ranked high by MDI but unimportant by permutation, MDI is misleading and permutation is the truth.

> **A question that often comes up here:** *"which single method should I report to a stakeholder?"* Permutation importance with error bars, plus the four-method heatmap behind it for context. Permutation is model-agnostic (works on any fitted estimator, not just trees), captures interaction effects rather than just marginal ones, and ships with built-in uncertainty (an SD across repeats that you can show as error bars). Drop-column is the gold standard but expensive; MDI is fast but biased toward high-cardinality features; linear-coef is the simplest reference but does not capture non-linear interactions. Permutation strikes the best balance for a poster or a memo.

**Key takeaway:** Methods agree when one feature dominates the signal; methods disagree when features are correlated or when one is high-cardinality. The disagreement is diagnostic — investigate it before reporting any single ranking.

---

## 8. Permutation Importance — Detail and Plotting

Permutation importance was already computed inside Section 7's four-method helper. This section visualizes it on its own with full error bars (the 10 repeats produce a per-feature SD) — the format you will use directly on the M3 milestone poster and any stakeholder memo.

Permutation importance is the right headline plot for three reasons. It is **model-agnostic** — the same procedure works on any fitted estimator, so you can use it later on the GBM in nb13 or the calibrated model in nb16 without rewriting code. It directly answers the question stakeholders actually want answered, which is *"how much does the prediction quality fall if this feature is unavailable?"* — not *"how often does this feature appear in a tree's split?"* And it ships with **built-in uncertainty** in the form of an SD across permutations, which becomes the error bar on the plot. Together those three properties make it the cleanest single visual to put in front of a non-technical audience.

In [ ]:
# Permutation importance bar charts with error bars — one per spine
fig, axes = plt.subplots(1, 2, figsize=(16, 9))
plot_importance_bars(perm_clf.importances_mean, X_train_clf.columns, axes[0],
                     errors=perm_clf.importances_std, color=CLF_COLOR,
                     title='Classification — permutation importance ± SD (10 repeats)', top_n=15)
plot_importance_bars(perm_reg.importances_mean, X_train_reg.columns, axes[1],
                     errors=perm_reg.importances_std, color=REG_COLOR,
                     title='Regression — permutation importance ± SD (10 repeats)', top_n=8)
fig.suptitle('Permutation importance — model-agnostic, with explicit uncertainty per feature',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The error bars are the part most readers underuse. A feature whose importance bar is short and whose error bar is wide enough to overlap zero is **not reliably important** — repeat the permutation enough times and it sometimes looks important and sometimes does not, which is the visual signature of pure noise dressed up as signal. Report the top features by mean **only when the SD is small enough that the rank is stable** under repeats.

For poster-ready figures, the rule of thumb is: include a feature in the importance discussion if its mean is at least 2× its SD. Below that threshold, the importance estimate is too noisy to defend to a stakeholder who will ask *"how sure are you?"*

> **A question that often comes up here:** *"why does permutation importance sometimes report negative values?"* Because the random shuffle can occasionally improve the score on a noisy feature — pure variance. A negative permutation-importance value is the algorithm's way of saying *"this feature carries no signal; the random labels are no worse than the real ones."* Treat negative entries as zeros and move on; they are not telling you the feature is *harmful*, only that there is no detectable signal in the data for it.

**Key takeaway:** Permutation importance is the headline plot for nb15 and your M3 poster. Always include the error bars; never report ranks for features whose mean falls below 2× SD. The error bars are what turn an importance ranking into a defensible one.

---

## 9. Comprehensive Model Comparison — Tree vs Forest vs Week-2 Reference

The closing section of the notebook puts the three models on the same CV-CI dot plot per spine: the **Week-2 reference baseline** that survived nb09's tuning sweeps (`LogReg(C=1.0)` for classification, OLS for regression), nb11's **single tree** at the depth that won its head-to-head (`max_depth=3` for classification, `max_depth=5` for regression), and today's **random forest** at default settings. This is the visual that nb14's selection ceremony will extend to a five-candidate field — today's three-candidate version is the dress rehearsal.

The verdict you should land on is binary per spine. On each spine, **does the forest's 95% CI clear the Week-2 reference's 95% CI by a visible margin?** If yes, the forest has earned displacement and ships. If no (CIs overlap), the linear baseline wins by simplicity under nb08's CI-overlap rule. The Reading-the-output cell at the end of the section names the two picks explicitly so you can defend them in the M3 milestone memo.

In [ ]:
# Comprehensive comparison — three models per spine.
# Single-tree depths match nb11's §7 winners (depth=3 clf, depth=5 reg).
# Week-2 references are exactly the pipelines from nb09's tuning sweeps.
single_tree_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED)
forest_clf_v    = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
single_tree_reg = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED)
forest_reg_v    = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)

clf_compare = {
    'Week-2 reference: LogReg(C=1.0)': cross_val_score(reference_clf,   X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Single Tree (depth=3, nb11)':     cross_val_score(single_tree_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (100 trees)':       cross_val_score(forest_clf_v,    X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_compare = {
    'Week-2 reference: OLS':           cross_val_score(reference_reg,   X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
    'Single Tree (depth=5, nb11)':     cross_val_score(single_tree_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
    'Random Forest (100 trees)':       cross_val_score(forest_reg_v,    X_train_reg, y_train_reg, cv=cv_reg, scoring='r2',      n_jobs=-1),
}

k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)
def _print_ci_summary(scores_dict, title):
    rows = []
    for name, s in scores_dict.items():
        mean = float(s.mean()); sd = float(s.std(ddof=1))
        half_w = t_crit * sd / np.sqrt(k)
        rows.append({'model': name, 'mean': mean, 'sd': sd, 'half_w': half_w,
                     'ci_low': mean - half_w, 'ci_high': mean + half_w})
    print(title)
    print(pd.DataFrame(rows).to_string(index=False))

_print_ci_summary(clf_compare, "=== CLASSIFICATION (5-fold CV ROC-AUC, mean ± 95% CI) ===")
print()
_print_ci_summary(reg_compare, "=== REGRESSION (5-fold CV R², mean ± 95% CI) ===")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_cv_ci(clf_compare, 'ROC-AUC', 'Classification — Tree vs Forest vs Week-2 reference', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_compare, 'R²',      'Regression — Tree vs Forest vs Week-2 reference',     axes[1], color=REG_COLOR)
fig.suptitle('Three-candidate dress rehearsal for nb14 — does the forest clear the linear reference?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

On the **regression spine**, the random forest's CV R² (typically ~0.80) clears the OLS reference (~0.60) by a CI-clear margin — well over 10 R-squared points of lift, with the forest's CI sitting entirely above the OLS CI. **Verdict: ship the random forest** for HomeValue Analytics' price-prediction tool. The forest has earned displacement of the Week-2 linear baseline by a defensible CI-clear margin, and the lift translates into thousands of dollars of better property-value predictions per listing.

On the **classification spine**, the random forest's CV ROC-AUC (typically ~0.98) is competitive with the LogReg(C=1.0) reference (~0.99), but the **CIs overlap**. The forest has not earned displacement on this dataset by the CI-overlap rule from nb08 — the two models are in a statistical tie. **Verdict: ship the Week-2 reference, LogReg(C=1.0)** for the State Health Department's screening tool. On a small, mostly-linearly-separable dataset, the simpler linear baseline wins by parsimony — and the four-method importance table you produced in Section 7 still gives you the interpretability story the oncologists asked for, because permutation importance is model-agnostic and you can apply it to the LogReg pipeline directly.

> **A question that often comes up here:** *"if the forest doesn't beat LogReg on classification, why was it worth running today?"* Three reasons. First, you produced the four-method importance reconciliation that the LogReg coefficients alone cannot give you on highly correlated features — that is a real interpretability deliverable independent of which model ships. Second, the forest's regression-spine win is decisive on the M3 milestone; one of your two spines now has a defensible non-linear champion. Third, **CI-overlap means statistical tie, which means simpler model wins** — that is a result, not a failure. Knowing that the data is essentially linear on the classification spine is itself useful information for the screening-tool stakeholders.

**Key takeaway:** Two spines, two verdicts. **Regression ships the random forest** (CI-clear lift over OLS); **classification ships the Week-2 LogReg(C=1.0)** (CIs overlap, simpler model wins). nb14 will run a more formal version of this comparison with five candidates per spine, but today's verdicts are already defensible and already documented.

---

## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification Forest

**Task:** Find the best `(n_estimators, max_features)` combination for `RandomForestClassifier` on Wisconsin breast cancer.

**Instructions:**
1. Sweep a 3×3 grid: `n_estimators ∈ [50, 100, 200]` × `max_features ∈ ['sqrt', 0.3, None]`.
2. For each combination, compute 5-fold CV ROC-AUC mean and SD using `cv_clf`.
3. Render as a heatmap with mean values annotated in each cell.
4. Pick the simplest combination whose CV mean is within one SD of the best (one-SE-rule).
5. Write 3 short findings: which dial moved the score most? Did the result confirm the `sqrt` default?

---

> 💡 **Gemini Prompt:** "Grid-search RandomForestClassifier(random_state=474, n_jobs=-1) over n_estimators=[50,100,200] × max_features=['sqrt',0.3,None] using 5-fold CV ROC-AUC on X_train_clf. Build a 3×3 heatmap of CV means with cell annotations and apply the one-SE rule to pick the simplest competitive combination."
>
> **After running, verify:**
> - [ ] Heatmap has 9 cells with mean values annotated
> - [ ] Best combination and one-SE-rule pick both reported
> - [ ] All cells use n_jobs=-1


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune RandomForestClassifier over (n_estimators, max_features) using 5-fold CV ROC-AUC.
# Apply the one-SE rule.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression Forest

**Task:** Find the best `(n_estimators, max_features)` combination for `RandomForestRegressor` on California Housing.

**Instructions:**
1. Sweep a 3×3 grid: `n_estimators ∈ [50, 100, 200]` × `max_features ∈ ['sqrt', 0.3, None]`.
2. For each combination, compute 3-fold CV R² mean and SD using `cv_reg_3` (3-fold for runtime).
3. Render as a heatmap with mean values annotated in each cell.
4. Pick the simplest combination whose CV mean is within one SD of the best (one-SE-rule).
5. Write 3 short findings: did the regression case prefer a different `max_features` than classification? Why?

---

> 💡 **Gemini Prompt:** "Grid-search RandomForestRegressor(random_state=474, n_jobs=-1) over n_estimators=[50,100,200] × max_features=['sqrt',0.3,None] using 3-fold CV R² on X_train_reg. Build a 3×3 heatmap of CV means with cell annotations and apply the one-SE rule. Report the best CV-RMSE in USD."
>
> **After running, verify:**
> - [ ] Heatmap has 9 cells with mean values annotated
> - [ ] Best combination and one-SE-rule pick both reported
> - [ ] Best CV-RMSE in USD printed


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune RandomForestRegressor over (n_estimators, max_features) using 3-fold CV R².
# Convert best CV-RMSE to USD; apply the one-SE rule.


## 10. Wrap-Up — Key Takeaways

**What landed today:**

1. **Bagging + random feature subsets reduce variance.** A forest of 100 trees has a tighter CV CI than a single tree on both spines — the variance-reduction-by-averaging math, made visible.
2. **OOB is a free second opinion on CV.** When OOB and CV agree, you have two estimates with different assumptions corroborating each other. When they disagree, suspect data dependence (groups, time).
3. **Four importance methods, four perspectives.** Linear-coef (interpretation), MDI (cheap), permutation (model-agnostic with uncertainty), drop-column (gold standard, expensive). Agreement is a strong signal; disagreement is information about the data.
4. **Forest beats OLS on regression by a CI-clear margin; ties LogReg on classification.** The Week-2 reference floor matters — and the forest only earns displacement on the regression spine.

**Bridge to nb13 — Gradient Boosting:**

Random forests reduce variance; gradient boosting reduces **bias**. Where forests fit many trees in parallel and average them, boosting fits trees sequentially — each one focused on the residuals (regression) or misclassifications (classification) of the previous ones. The result is often a CV-score lift of several points over the random forest, especially on regression with structure that a single deeper tree can capture but a shallow forest dilutes.

The same dual-spine pattern continues: `GradientBoostingClassifier` on Wisconsin breast cancer, `GradientBoostingRegressor` on California Housing, paired diagnostics at every step, both compared against the Week-2 reference floor. Bring today's tuning muscle memory — boosting needs `learning_rate`, `n_estimators`, and `max_depth` tuned together rather than independently.

> **A question that often comes up at this point:** *"does gradient boosting always beat random forest?"* On most tabular datasets, yes — by a small margin. But the trade is real: boosting is sequential (slower to fit, harder to parallelize), more sensitive to hyperparameters (a wrong learning rate can wreck the model), and more prone to overfitting if you don't early-stop. nb13 walks both the wins and the trade-offs.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (RF tuning, classification) and Exercise 2 (RF tuning, regression).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 12 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a tuning heatmap with the chosen combination starred
- [ ] The four-method importance heatmap renders for both spines
- [ ] All figures render (none broken)
- [ ] Both `_clf` and `_reg` variable namespaces stay disjoint (no `NameError`)

### Next Step:

- **Notebook 13** — Gradient Boosting (Day 13)

---

<center>

**Thank you!**

</center>